<a href="https://colab.research.google.com/github/aaradhyasirohi/MachineLearning/blob/main/digital_recognition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# =====================================
# Backpropagation using NumPy (MNIST)
# =====================================

import numpy as np
from keras.datasets import mnist
from sklearn.preprocessing import OneHotEncoder

# 1. Load dataset
(x_train, y_train), (x_test, y_test) = mnist.load_data()

# Normalize
x_train = x_train.reshape(-1, 784) / 255.0
x_test = x_test.reshape(-1, 784) / 255.0

# One-hot encoding
encoder = OneHotEncoder(sparse_output=False)
y_train = encoder.fit_transform(y_train.reshape(-1, 1))
y_test = encoder.transform(y_test.reshape(-1, 1))

# 2. Initialize parameters
input_size = 784
hidden_size = 128
output_size = 10

np.random.seed(42)
W1 = np.random.randn(input_size, hidden_size) * 0.01
b1 = np.zeros((1, hidden_size))
W2 = np.random.randn(hidden_size, output_size) * 0.01
b2 = np.zeros((1, output_size))

# 3. Activation functions
def relu(Z):
    return np.maximum(0, Z)

def relu_derivative(Z):
    return Z > 0

def softmax(Z):
    expZ = np.exp(Z - np.max(Z, axis=1, keepdims=True))
    return expZ / np.sum(expZ, axis=1, keepdims=True)

# 4. Training
epochs = 10
lr = 0.01
batch_size = 64

for epoch in range(epochs):
    for i in range(0, x_train.shape[0], batch_size):
        X = x_train[i:i+batch_size]
        Y = y_train[i:i+batch_size]

        # ---- Forward pass ----
        Z1 = np.dot(X, W1) + b1
        A1 = relu(Z1)

        Z2 = np.dot(A1, W2) + b2
        A2 = softmax(Z2)

        # ---- Loss (cross-entropy) ----
        m = Y.shape[0]
        loss = -np.sum(Y * np.log(A2 + 1e-8)) / m

        # ---- Backpropagation ----
        dZ2 = A2 - Y
        dW2 = np.dot(A1.T, dZ2) / m
        db2 = np.sum(dZ2, axis=0, keepdims=True) / m

        dA1 = np.dot(dZ2, W2.T)
        dZ1 = dA1 * relu_derivative(Z1)
        dW1 = np.dot(X.T, dZ1) / m
        db1 = np.sum(dZ1, axis=0, keepdims=True) / m

        # ---- Update weights ----
        W1 -= lr * dW1
        b1 -= lr * db1
        W2 -= lr * dW2
        b2 -= lr * db2

    print(f"Epoch {epoch+1}, Loss: {loss:.4f}")

# 5. Evaluation
Z1 = np.dot(x_test, W1) + b1
A1 = relu(Z1)
Z2 = np.dot(A1, W2) + b2
A2 = softmax(Z2)

predictions = np.argmax(A2, axis=1)
accuracy = np.mean(predictions == np.argmax(y_test, axis=1))

print("\nTest Accuracy:", accuracy)

Epoch 1, Loss: 1.0042
Epoch 2, Loss: 0.3878
Epoch 3, Loss: 0.2590
Epoch 4, Loss: 0.2055
Epoch 5, Loss: 0.1764
Epoch 6, Loss: 0.1583
Epoch 7, Loss: 0.1454
Epoch 8, Loss: 0.1350
Epoch 9, Loss: 0.1260
Epoch 10, Loss: 0.1192

Test Accuracy: 0.9243
